In [4]:
bk_zip_codes = ['11201', '11206', '11207', '11208', '11209', '11202', '11203', '11204', '11205', '11210', '11211', '11212', '11213', '11218', '11219', '11220', '11221', '11222', '11223', '11224', '11225', '11214', '11215', '11216', '11217', '11226', '11228', '11229', '11230', '11235', '11236', '11237', '11238', '11245', '11247', '11249', '11256', '11231', '11232', '11233', '11234', '11239', '11241', '11242', '11243', '11251', '11252']
len(bk_zip_codes)

47

In [5]:
import os
from dotenv import load_dotenv
load_dotenv()
import requests
census_api_key = os.environ.get("CENSUS_API_KEY")
import pandas as pd
import time

In [6]:
import time
import pandas as pd
import requests

# ZIP code batches
bk_zip_codes = ['11201', '11206', '11207', '11208', '11209', '11202', '11203', '11204', '11205',
                '11210', '11211', '11212', '11213', '11218', '11219', '11220', '11221', '11222',
                '11223', '11224', '11225', '11214', '11215', '11216', '11217', '11226', '11228',
                '11229', '11230', '11235', '11236', '11237', '11238', '11245', '11247', '11249',
                '11256', '11231', '11232', '11233', '11234', '11239', '11241', '11242', '11243',
                '11251', '11252']

batch_size = 38
zip_batches = [bk_zip_codes[i:i + batch_size] for i in range(0, len(bk_zip_codes), batch_size)]
years = [2022, 2023]

# Store data from both years
all_data = pd.DataFrame()

for year in years:
    print(f"\n📅 Processing year: {year}")
    for i, batch in enumerate(zip_batches, start=1):
        zcta_str = ",".join(batch)
        url = f"https://api.census.gov/data/{year}/acs/acs5/subject?get=NAME,group(S1901)&for=zip%20code%20tabulation%20area:{zcta_str}&key={census_api_key}"

        print(f"🌐 Requesting batch {i} for year {year} with {len(batch)} ZCTAs...")

        try:
            response = requests.get(url, timeout=10)
            response.raise_for_status()
            data = response.json()

            if len(data) <= 1:
                print(f"⚠️ No data returned for batch {i}, year {year}")
            else:
                df = pd.DataFrame(data[1:], columns=data[0])
                df["year"] = year  # add a year column for distinction
                all_data = pd.concat([all_data, df], ignore_index=True)
                print(f"✅ Added {len(df)} rows from batch {i}, year {year}")
        except requests.exceptions.RequestException as e:
            print(f"❌ Request failed for batch {i}, year {year}: {e}")

        time.sleep(2)  # respect API limits

# Optionally reset index
all_data.reset_index(drop=True, inplace=True)



📅 Processing year: 2022
🌐 Requesting batch 1 for year 2022 with 38 ZCTAs...
✅ Added 34 rows from batch 1, year 2022
🌐 Requesting batch 2 for year 2022 with 9 ZCTAs...
✅ Added 4 rows from batch 2, year 2022

📅 Processing year: 2023
🌐 Requesting batch 1 for year 2023 with 38 ZCTAs...
✅ Added 34 rows from batch 1, year 2023
🌐 Requesting batch 2 for year 2023 with 9 ZCTAs...
✅ Added 4 rows from batch 2, year 2023


In [7]:


# Get variable definitions
vars_url = "https://api.census.gov/data/2023/acs/acs5/subject/variables.json"
vars_response = requests.get(vars_url, timeout=10).json()
variables = vars_response['variables']

# match labels for S1901 variables
s1901_labels = {
    var: info['label'] 
    for var, info in variables.items() 
    if var.startswith("S1901_C01_") and var.endswith("E")
}

s1901_labels = s1901_labels.items()
s1901_labels = pd.DataFrame(s1901_labels)
s1901_labels.rename(columns={0: 'code_label', 1: 'readable_label'}, inplace=True)
s1901_labels.columns

s1901_labels.to_csv("s1901_labels.csv", index=False)


In [8]:
col_names = list(s1901_labels['code_label'])
columns_to_select = col_names + ['GEO_ID', 'zip code tabulation area']
filtered_data = all_data[columns_to_select]
filtered_data = filtered_data.rename(columns={"zip code tabulation area": "ZIP"})

In [9]:
import pandas as pd

def rename_columns_with_labels(data_df, labels_df):
    """
    Rename columns in data_df using human-readable labels from labels_df
    
    Args:
        data_df: DataFrame with code labels as column names
        labels_df: DataFrame with code_label and readable_label columns
    
    Returns:
        DataFrame with renamed columns
    """
    # Check if the expected columns exist in labels_df
    if 'code_label' not in labels_df.columns or 'readable_label' not in labels_df.columns:
        # Print the actual column names for debugging
        print(f"Available columns in labels dataframe: {labels_df.columns.tolist()}")
        raise ValueError("Labels dataframe must contain 'code_label' and 'readable_label' columns")
    
    # Clean the readable labels by replacing '!!' with a single space
    cleaned_labels = labels_df['readable_label'].str.replace('!!', ' ')
    
    # Create a dictionary mapping code labels to cleaned readable labels
    label_dict = dict(zip(labels_df['code_label'], cleaned_labels))
    
    # Create a copy of the original dataframe
    renamed_df = data_df.copy()
    
    # Rename columns that have corresponding readable labels
    columns_to_rename = {col: label_dict.get(col, col) for col in data_df.columns if col in label_dict}
    renamed_df = renamed_df.rename(columns=columns_to_rename)
    
    return renamed_df

renamed_df = rename_columns_with_labels(filtered_data, s1901_labels)

# Display the result
renamed_df.head()

,Estimate Households PERCENT ALLOCATED Nonfamily income in the past 12 months,Estimate Households PERCENT ALLOCATED Family income in the past 12 months,Estimate Households PERCENT ALLOCATED Household income in the past 12 months,Estimate Households Mean income (dollars),Estimate Households Median income (dollars),"Estimate Households Total $200,000 or more","Estimate Households Total $150,000 to $199,999","Estimate Households Total $100,000 to $149,999","Estimate Households Total $75,000 to $99,999","Estimate Households Total $50,000 to $74,999","Estimate Households Total $35,000 to $49,999","Estimate Households Total $25,000 to $34,999","Estimate Households Total $15,000 to $24,999","Estimate Households Total $10,000 to $14,999","Estimate Households Total Less than $10,000",Estimate Households Total,GEO_ID,ZIP
0,-888888888,-888888888,31.5,232020,163310,40.7,12.4,13.8,5.8,7.0,4.7,3.3,3.8,2.7,5.7,32227,860Z200US11201,11201
1,-888888888,-888888888,59.5,89068,68006,7.9,8.3,16.2,12.8,16.8,11.6,8.1,6.1,5.1,7.1,29411,860Z200US11203,11203
2,-888888888,-888888888,50.4,83981,64172,6.1,8.8,14.4,14.1,18.3,10.6,8.8,9.1,5.2,4.6,25118,860Z200US11204,11204
3,-888888888,-888888888,43.3,132369,74839,19.9,8.6,14.5,6.9,13.4,9.7,6.4,7.0,6.2,7.4,16861,860Z200US11205,11205
4,-888888888,-888888888,45.6,81598,51507,8.9,6.7,12.4,10.8,12.2,9.3,9.3,12.3,7.7,10.3,32698,860Z200US11206,11206


In [10]:
# BK DATA FOR 2023 INCOME BREAKDOWN BY ZIPCODE
bk_income_data_by_zip = renamed_df

bk_income_data_by_zip.to_csv("bk_income_data_by_zip.csv", index=False)

In [11]:
bk_income_data_by_zip.head()

,Estimate Households PERCENT ALLOCATED Nonfamily income in the past 12 months,Estimate Households PERCENT ALLOCATED Family income in the past 12 months,Estimate Households PERCENT ALLOCATED Household income in the past 12 months,Estimate Households Mean income (dollars),Estimate Households Median income (dollars),"Estimate Households Total $200,000 or more","Estimate Households Total $150,000 to $199,999","Estimate Households Total $100,000 to $149,999","Estimate Households Total $75,000 to $99,999","Estimate Households Total $50,000 to $74,999","Estimate Households Total $35,000 to $49,999","Estimate Households Total $25,000 to $34,999","Estimate Households Total $15,000 to $24,999","Estimate Households Total $10,000 to $14,999","Estimate Households Total Less than $10,000",Estimate Households Total,GEO_ID,ZIP
0,-888888888,-888888888,31.5,232020,163310,40.7,12.4,13.8,5.8,7.0,4.7,3.3,3.8,2.7,5.7,32227,860Z200US11201,11201
1,-888888888,-888888888,59.5,89068,68006,7.9,8.3,16.2,12.8,16.8,11.6,8.1,6.1,5.1,7.1,29411,860Z200US11203,11203
2,-888888888,-888888888,50.4,83981,64172,6.1,8.8,14.4,14.1,18.3,10.6,8.8,9.1,5.2,4.6,25118,860Z200US11204,11204
3,-888888888,-888888888,43.3,132369,74839,19.9,8.6,14.5,6.9,13.4,9.7,6.4,7.0,6.2,7.4,16861,860Z200US11205,11205
4,-888888888,-888888888,45.6,81598,51507,8.9,6.7,12.4,10.8,12.2,9.3,9.3,12.3,7.7,10.3,32698,860Z200US11206,11206


In [15]:
# Convert all relevant columns to numeric, coercing errors
bk_income_data_by_zip[bk_income_data_by_zip.columns] = bk_income_data_by_zip[bk_income_data_by_zip.columns].apply(pd.to_numeric, errors='coerce')
bk_income_data_by_zip.head()


,Estimate Households PERCENT ALLOCATED Nonfamily income in the past 12 months,Estimate Households PERCENT ALLOCATED Family income in the past 12 months,Estimate Households PERCENT ALLOCATED Household income in the past 12 months,Estimate Households Mean income (dollars),Estimate Households Median income (dollars),"Estimate Households Total $200,000 or more","Estimate Households Total $150,000 to $199,999","Estimate Households Total $100,000 to $149,999","Estimate Households Total $75,000 to $99,999","Estimate Households Total $50,000 to $74,999",...,"Estimate Households Total $25,000 to $34,999","Estimate Households Total $15,000 to $24,999","Estimate Households Total $10,000 to $14,999","Estimate Households Total Less than $10,000",Estimate Households Total,GEO_ID,ZIP,pct_income_below_25k,pct_income_25k_75k,pct_income_75k_plus
0,-888888888,-888888888,31.5,232020,163310,40.7,12.4,13.8,5.8,7.0,...,3.3,3.8,2.7,5.7,32227,NaN,11201,0.000379,0.000465,0.002256
1,-888888888,-888888888,59.5,89068,68006,7.9,8.3,16.2,12.8,16.8,...,8.1,6.1,5.1,7.1,29411,NaN,11203,0.000622,0.001241,0.001537
2,-888888888,-888888888,50.4,83981,64172,6.1,8.8,14.4,14.1,18.3,...,8.8,9.1,5.2,4.6,25118,NaN,11204,0.000752,0.001501,0.001728
3,-888888888,-888888888,43.3,132369,74839,19.9,8.6,14.5,6.9,13.4,...,6.4,7.0,6.2,7.4,16861,NaN,11205,0.001222,0.001750,0.002959
4,-888888888,-888888888,45.6,81598,51507,8.9,6.7,12.4,10.8,12.2,...,9.3,12.3,7.7,10.3,32698,NaN,11206,0.000927,0.000942,0.001187


In [ ]:
# Define your income groups as a dictionary
bracket_cols = [
    'Estimate Households Total $200,000 or more',
    'Estimate Households Total $150,000 to $199,999',
    'Estimate Households Total $100,000 to $149,999',
    'Estimate Households Total $75,000 to $99,999',
    'Estimate Households Total $50,000 to $74,999',
    'Estimate Households Total $35,000 to $49,999',
    'Estimate Households Total $25,000 to $34,999',
    'Estimate Households Total $15,000 to $24,999',
    'Estimate Households Total $10,000 to $14,999',
    'Estimate Households Total Less than $10,000'
]



for col in bracket_cols:
    bk_income_data_by_zip[col] = pd.to_numeric(bk_income_data_by_zip[col], errors='coerce')

# Step 2: Create the consolidated income brackets
# Above $75k (sum of $75k+ categories)
bk_income_data_by_zip['Income Above $75k (%)'] = bk_income_data_by_zip[[
    'Estimate Households Total $200,000 or more',
    'Estimate Households Total $150,000 to $199,999',
    'Estimate Households Total $100,000 to $149,999',
    'Estimate Households Total $75,000 to $99,999'
]].sum(axis=1)

# $25k-$75k (sum of $25k-$75k categories)
bk_income_data_by_zip['Income $25k-$75k (%)'] = bk_income_data_by_zip[[
    'Estimate Households Total $50,000 to $74,999',
    'Estimate Households Total $35,000 to $49,999',
    'Estimate Households Total $25,000 to $34,999'
]].sum(axis=1)

# Below $25k (sum of <$25k categories)
bk_income_data_by_zip['Income Below $25k (%)'] = bk_income_data_by_zip[[
    'Estimate Households Total $15,000 to $24,999',
    'Estimate Households Total $10,000 to $14,999',
    'Estimate Households Total Less than $10,000'
]].sum(axis=1)

# Step 3: Calculate the absolute household counts using the percentages
bk_income_data_by_zip['Income Above $75k (count)'] = (bk_income_data_by_zip['Income Above $75k (%)'] / 100) * bk_income_data_by_zip['Estimate Households Total']
bk_income_data_by_zip['Income $25k-$75k (count)'] = (bk_income_data_by_zip['Income $25k-$75k (%)'] / 100) * bk_income_data_by_zip['Estimate Households Total']
bk_income_data_by_zip['Income Below $25k (count)'] = (bk_income_data_by_zip['Income Below $25k (%)'] / 100) * bk_income_data_by_zip['Estimate Households Total']

# Round the counts to integers since they represent households
bk_income_data_by_zip['Income Above $75k (count)'] = bk_income_data_by_zip['Income Above $75k (count)'].round().astype(int)
bk_income_data_by_zip['Income $25k-$75k (count)'] = bk_income_data_by_zip['Income $25k-$75k (count)'].round().astype(int)
bk_income_data_by_zip['Income Below $25k (count)'] = bk_income_data_by_zip['Income Below $25k (count)'].round().astype(int)

# Step 4: Create a simplified DataFrame with just the needed columns
simplified_df = bk_income_data_by_zip[['ZIP', 
                   'Estimate Households Total',
                   'Income Above $75k (%)', 'Income $25k-$75k (%)', 'Income Below $25k (%)',
                   'Income Above $75k (count)', 'Income $25k-$75k (count)', 'Income Below $25k (count)']]

# Display the simplified DataFrame
print(simplified_df)

# Verify that the percentages still sum to approximately 100%
print("\nVerification - Sum of percentage columns:")
percentage_sum = simplified_df[['Income Above $75k (%)', 'Income $25k-$75k (%)', 'Income Below $25k (%)']].sum(axis=1)
# print(percentage_sum)

# Verify that the counts sum to the total households
# print("\nVerification - Sum of count columns vs total households:")
# count_sum = simplified_df[['Income Above $75k (count)', 'Income $25k-$75k (count)', 'Income Below $25k (count)']].sum(axis=1)
# total_diff = simplified_df['Estimate Households Total'] - count_sum
# pd.DataFrame({'Total Households': simplified_df['Estimate Households Total'], 
#                     'Sum of Counts': count_sum,
#                     'Difference': total_diff})



      ZIP  Estimate Households Total  Income Above $75k (%)  \
0   11201                      32227                   72.7   
1   11203                      29411                   45.2   
2   11204                      25118                   43.4   
3   11205                      16861                   49.9   
4   11206                      32698                   38.8   
..    ...                        ...                    ...   
71  11249                      17621                   64.4   
72  11232                       9336                   55.8   
73  11233                      32784                   43.6   
74  11234                      32178                   59.1   
75  11239                       8320                   23.7   

    Income $25k-$75k (%)  Income Below $25k (%)  Income Above $75k (count)  \
0                   15.0                   12.2                      23429   
1                   36.5                   18.3                      13294   
2        

,Total Households,Sum of Counts,Difference
0,32227,32195,32
1,29411,29411,0
2,25118,25117,1
3,16861,16861,0
4,32698,32665,33
...,...,...,...
71,17621,17621,0
72,9336,9326,10
73,32784,32752,32
74,32178,32113,65


In [18]:
simplified_df.head()

,ZIP,Estimate Households Total,Income Above $75k (%),Income $25k-$75k (%),Income Below $25k (%),Income Above $75k (count),Income $25k-$75k (count),Income Below $25k (count)
0,11201,32227,72.7,15.0,12.2,23429,4834,3932
1,11203,29411,45.2,36.5,18.3,13294,10735,5382
2,11204,25118,43.4,37.7,18.9,10901,9469,4747
3,11205,16861,49.9,29.5,20.6,8414,4974,3473
4,11206,32698,38.8,30.8,30.3,12687,10071,9907
